# Walkthrough do pipeline de extração

Demonstra, passo a passo, as técnicas clássicas usadas para extrair o grafo de conhecimento de um caso clínico do MultiCaRe: pré-processamento, NER por gazetteer, extração de valores/unidades e regras de relação, terminando no grafo canônico e nas 3 visões de abstração.

In [ ]:
import sys
sys.path.insert(0, "../../src")

import pandas as pd
from kg_extraction.data.make_dataset import load_cases
from kg_extraction.features.preprocessing import split_sentences, tokenize
from kg_extraction.graph.canonical_graph import extract_case_graph
from kg_extraction.graph.views import derive_view

cases = load_cases()
case_row = cases.iloc[0]
print(case_row["case_id"], case_row["age"], case_row["gender"])
print(case_row["case_text"][:500])

## 1. Pré-processamento (tokenização + segmentação de sentenças)

In [ ]:
sentences = split_sentences(case_row["case_text"])
print(f"{len(sentences)} sentenças")
for s in sentences[:3]:
    print("-", s)

## 2. Extração do grafo canônico (NER por gazetteer + regex + regras de relação)

In [ ]:
nodes, edges = extract_case_graph(case_row)
nodes_df, edges_df = pd.DataFrame(nodes), pd.DataFrame(edges)
print(f"{len(nodes_df)} nos, {len(edges_df)} arestas")
nodes_df.head(10)

## 3. Derivando os 3 niveis de abstracao do mesmo grafo canonico

In [ ]:
for level in ("basic", "intermediate", "detailed"):
    n, e = derive_view(level, nodes_df, edges_df, case_row["case_id"])
    print(f"{level:14s} -> {len(n):3d} nos, {len(e):3d} arestas, tipos={sorted(n.type.unique())}")